# Forecast Metrics Introduction

This notebook introduces the most commonly used forecast evaluation metrics.
Understanding these metrics is essential for assessing how well your forecasting models perform.

**Topics covered:**
- **MAE** - Mean Absolute Error
- **RMSE** - Root Mean Square Error
- **MAPE** - Mean Absolute Percentage Error
- **MASE** - Mean Absolute Scaled Error

We will use Brazilian macroeconomic data to illustrate each metric with practical examples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.metrics import mae, rmse, mape, mase

import sys
sys.path.insert(0, "../..")
from utils.helpers import load_macro_brazil, load_macro_us, plot_series

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Loading Data

We use two macroeconomic datasets generated for this tutorial:

- **macro_brazil.csv**: Brazilian macro indicators (GDP growth, inflation, interest rate, unemployment, exchange rate)
- **macro_us.csv**: US macro indicators (GDP growth, CPI inflation, Fed Funds rate, unemployment)

Let's start by loading and exploring the Brazilian dataset.

In [ ]:
df_brazil = load_macro_brazil()
print("Shape:", df_brazil.shape)
print()
df_brazil.head()

In [ ]:
df_brazil.info()

In [ ]:
# Create a simple "forecast" by shifting gdp_growth forward by 1 period (naive forecast)
gdp = df_brazil["gdp_growth"].dropna()
actual = gdp.iloc[1:].values
predicted = gdp.iloc[:-1].values  # naive: forecast = previous value
training = gdp.iloc[:60].values   # first 60 obs for MASE scaling

print(f"Evaluation period: {len(actual)} observations")
print(f"Actual (first 5):    {actual[:5].round(4)}")
print(f"Predicted (first 5): {predicted[:5].round(4)}")

## 2. MAE - Mean Absolute Error

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

**Interpretation:**
- MAE measures the average magnitude of errors, ignoring their direction.
- It is in the **same units** as the original data, making it easy to interpret.
- MAE treats all errors equally (unlike RMSE which penalizes large errors more).
- A MAE of 0.5 for GDP growth means the forecast is off by 0.5 percentage points on average.

In [ ]:
# Manual calculation
mae_manual = np.mean(np.abs(actual - predicted))
print(f"MAE (manual):      {mae_manual:.4f}")

# Using forecastbox
mae_fb = mae(actual, predicted)
print(f"MAE (forecastbox): {mae_fb:.4f}")

# Verify they match
assert np.isclose(mae_manual, mae_fb), "Values should match!"
print("\nInterpretation: the naive forecast is off by", f"{mae_fb:.4f}",
      "percentage points on average.")

## 3. RMSE - Root Mean Square Error

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

**Interpretation:**
- RMSE also measures error magnitude in the same units as the data.
- It **penalizes large errors more** than MAE due to squaring.
- RMSE >= MAE always. When RMSE >> MAE, it indicates the presence of large outlier errors.
- Preferred when large errors are particularly costly (e.g., financial risk).

In [ ]:
# Manual calculation
rmse_manual = np.sqrt(np.mean((actual - predicted) ** 2))
print(f"RMSE (manual):      {rmse_manual:.4f}")

# Using forecastbox
rmse_fb = rmse(actual, predicted)
print(f"RMSE (forecastbox): {rmse_fb:.4f}")

assert np.isclose(rmse_manual, rmse_fb)

# Compare with MAE
print(f"\nMAE:  {mae_fb:.4f}")
print(f"RMSE: {rmse_fb:.4f}")
print(f"RMSE/MAE ratio: {rmse_fb / mae_fb:.2f}")
print("A ratio close to 1.0 means errors are uniform; higher means some large errors exist.")

## 4. MAPE - Mean Absolute Percentage Error

$$MAPE = \frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

**Interpretation:**
- MAPE expresses error as a **percentage** of the actual value.
- Scale-independent: useful for comparing forecasts across different series.

**Limitations:**
- **Undefined when actual values are zero** (division by zero).
- **Asymmetric**: penalizes over-forecasting more than under-forecasting.
- Can be extremely large when actual values are close to zero.
- Not suitable for data that crosses zero (e.g., growth rates that can be negative).

In [ ]:
# MAPE for inflation (positive values, MAPE works well)
inflation = df_brazil["inflation"].dropna()
actual_inf = inflation.iloc[1:].values
predicted_inf = inflation.iloc[:-1].values

mape_inf = mape(actual_inf, predicted_inf)
print(f"MAPE for inflation (naive): {mape_inf:.2f}%")

# MAPE for GDP growth (can have values near zero - problematic!)
mape_gdp = mape(actual, predicted)
print(f"MAPE for GDP growth (naive): {mape_gdp:.2f}%")
print("\nNote: MAPE can be very high or infinite when actual values are near zero.")

## 5. MASE - Mean Absolute Scaled Error

$$MASE = \frac{\frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t - y_{t-1}|}$$

**Interpretation:**
- MASE scales the forecast error by the in-sample MAE of a **naive forecast**.
- **MASE < 1**: the model outperforms the naive baseline.
- **MASE = 1**: the model performs the same as the naive baseline.
- **MASE > 1**: the model is worse than the naive baseline.

**Advantages over MAPE:**
- Well-defined even when actual values are zero.
- Symmetric (no bias towards over- or under-forecasting).
- Scale-independent: can compare across different series.
- Recommended by Hyndman & Koehler (2006) as a general-purpose metric.

In [ ]:
# MASE requires the training series for scaling
# Split: first 80% for training, last 20% for evaluation
n = len(gdp)
split = int(0.8 * n)

train_series = gdp.iloc[:split].values
test_actual = gdp.iloc[split:].values
test_naive = gdp.iloc[split - 1 : -1].values  # naive forecast for test period

mase_value = mase(test_actual, test_naive, training_series=train_series)
print(f"MASE (naive forecast): {mase_value:.4f}")
print("\nBy definition, the naive forecast should have MASE close to 1.0")
print("(not exactly 1.0 because train and test MAE differ).")

# A "better" forecast: average of last 3 values
test_sma3 = pd.Series(gdp.values).rolling(3).mean().dropna().iloc[split - 3:].values[:len(test_actual)]
mase_sma3 = mase(test_actual, test_sma3, training_series=train_series)
print(f"\nMASE (SMA-3 forecast): {mase_sma3:.4f}")
if mase_sma3 < 1.0:
    print("SMA-3 beats the naive baseline!")
else:
    print("SMA-3 does not beat the naive baseline.")

## 6. Comparing Metrics

Each metric has strengths and weaknesses. Here is a guide for choosing:

| Metric | Scale-dependent? | Handles zeros? | Penalizes large errors? | Best for |
|--------|:---:|:---:|:---:|---|
| MAE | Yes | Yes | Equally | General-purpose, interpretable |
| RMSE | Yes | Yes | More heavily | When large errors are costly |
| MAPE | No (%) | **No** | Equally | Comparing across scales (positive data) |
| MASE | No (scaled) | Yes | Equally | Comparing across series, benchmarking |

In [ ]:
# Compare metrics across different forecast methods for GDP growth
n = len(gdp)
split = int(0.8 * n)
train = gdp.iloc[:split]
test = gdp.iloc[split:]

# Different forecasting approaches
# For SMA, use the mean of last k training values as a constant forecast (avoids length mismatch)
forecasts = {
    "Naive (last value)": np.full(len(test), train.iloc[-1]),
    "Mean": np.full(len(test), train.mean()),
    "SMA-6": np.full(len(test), train.iloc[-6:].mean()),
    "SMA-12": np.full(len(test), train.iloc[-12:].mean()),
}

# Build comparison table
rows = []
for name, pred in forecasts.items():
    rows.append({
        "Method": name,
        "MAE": mae(test.values, pred),
        "RMSE": rmse(test.values, pred),
        "MAPE": mape(test.values, pred),
        "MASE": mase(test.values, pred, training_series=train.values),
    })

comparison = pd.DataFrame(rows).set_index("Method")
print("Metric Comparison for GDP Growth Forecasts")
print("=" * 60)
comparison.round(4)

## Exercise 1: Calculate all metrics for US data

Load the US macroeconomic dataset (`macro_us.csv`) and compute MAE, RMSE, MAPE, and MASE
for a naive forecast of `gdp_growth`. Use an 80/20 train/test split.

In [ ]:
# SOLUTION: Exercise 1 - Load macro_us.csv and compute MAE, RMSE, MAPE, MASE for gdp_growth

# Step 1: Load US macroeconomic data
df_us = load_macro_us()
print("US dataset shape:", df_us.shape)
print("Columns:", df_us.columns.tolist())
print()

# Step 2: Extract GDP growth and create train/test split (80/20)
gdp_us = df_us["gdp_growth"].dropna()
n_us = len(gdp_us)
split_us = int(0.8 * n_us)

train_us = gdp_us.iloc[:split_us]
test_us = gdp_us.iloc[split_us:]
print(f"Total observations: {n_us}")
print(f"Training: {split_us}, Test: {n_us - split_us}")

# Step 3: Create naive forecast (repeat last training value)
naive_pred_us = np.full(len(test_us), train_us.iloc[-1])
print(f"\nNaive forecast value (last training obs): {train_us.iloc[-1]:.4f}")

# Step 4: Compute all four metrics
mae_us = mae(test_us.values, naive_pred_us)
rmse_us = rmse(test_us.values, naive_pred_us)
mape_us = mape(test_us.values, naive_pred_us)
mase_us = mase(test_us.values, naive_pred_us, training_series=train_us.values)

print(f"\n--- Metrics for Naive Forecast on US GDP Growth ---")
print(f"MAE:  {mae_us:.4f}")     # Expected: ~0.1974
print(f"RMSE: {rmse_us:.4f}")    # Expected: ~0.2209
print(f"MAPE: {mape_us:.4f}%")   # Expected: ~108.19 (very high due to values near zero!)
print(f"MASE: {mase_us:.4f}")    # Expected: ~1.3204

# Step 5: Interpretation
print("\n--- Interpretation ---")
print(f"MAE of {mae_us:.4f} means the naive forecast is off by ~{mae_us:.2f} pp on average.")
print(f"RMSE/MAE ratio: {rmse_us / mae_us:.2f} -- close to 1, errors are relatively uniform.")
print(f"MAPE of {mape_us:.2f}% is extremely high -- this is because GDP growth values")
print(f"  can be very close to zero, making percentage errors blow up.")
print(f"MASE of {mase_us:.4f} > 1 -- the naive forecast on the test set is worse than")
print(f"  the in-sample naive baseline, indicating the test period is harder to forecast.")

## Exercise 2: Which metric is most appropriate for inflation forecasting? Why?

Consider the properties of inflation data (always positive, can be close to zero in low-inflation
environments, sometimes has large spikes). Load both datasets and experiment with different metrics
to support your answer.

In [ ]:
# SOLUTION: Exercise 2 - Which metric is most appropriate for inflation forecasting?

# Let's analyze inflation data from both countries to understand the properties
df_us = load_macro_us()
df_brazil = load_macro_brazil()

inflation_br = df_brazil["inflation"].dropna()
cpi_us = df_us["cpi_inflation"].dropna()

print("=== Inflation Data Properties ===")
print(f"\nBrazil Inflation:")
print(f"  Mean:  {inflation_br.mean():.4f}")
print(f"  Std:   {inflation_br.std():.4f}")
print(f"  Min:   {inflation_br.min():.4f}")
print(f"  Max:   {inflation_br.max():.4f}")
print(f"  Near zero (< 0.05): {(inflation_br.abs() < 0.05).sum()} observations")

print(f"\nUS CPI Inflation:")
print(f"  Mean:  {cpi_us.mean():.4f}")
print(f"  Std:   {cpi_us.std():.4f}")
print(f"  Min:   {cpi_us.min():.4f}")
print(f"  Max:   {cpi_us.max():.4f}")
print(f"  Near zero (< 0.05): {(cpi_us.abs() < 0.05).sum()} observations")

# Compute metrics for naive forecasts on both inflation series
for name, series in [("Brazil Inflation", inflation_br), ("US CPI Inflation", cpi_us)]:
    n = len(series)
    split = int(0.8 * n)
    train = series.iloc[:split]
    test = series.iloc[split:]
    naive_pred = np.full(len(test), train.iloc[-1])

    print(f"\n--- {name}: Naive Forecast Metrics ---")
    print(f"  MAE:  {mae(test.values, naive_pred):.4f}")
    print(f"  RMSE: {rmse(test.values, naive_pred):.4f}")
    print(f"  MAPE: {mape(test.values, naive_pred):.4f}%")
    print(f"  MASE: {mase(test.values, naive_pred, training_series=train.values):.4f}")

# Analysis and recommendation
print("\n" + "=" * 60)
print("ANALYSIS: Which metric is best for inflation forecasting?")
print("=" * 60)
print("""
For inflation forecasting, MASE is the most appropriate metric. Here's why:

1. MAPE has serious problems with inflation data:
   - Inflation values can approach zero (especially in low-inflation environments).
   - When actual values are near zero, MAPE explodes (division by near-zero).
   - MAPE is asymmetric: it penalizes over-forecasting more than under-forecasting.
   - In our US data, MAPE can become unreliable when CPI inflation dips low.

2. MAE and RMSE are scale-dependent:
   - MAE/RMSE values depend on the scale of the data.
   - You cannot compare MAE across Brazil inflation (~0.5%) and US inflation (~0.2%).
   - They are good for comparing models on the SAME series, but not across series.

3. MASE is the recommended choice (Hyndman & Koehler, 2006):
   - Well-defined even when inflation is zero (no division by actual values).
   - Scale-independent: can compare forecast quality across countries/series.
   - Symmetric: treats over- and under-forecasting equally.
   - Interpretable: MASE < 1 means "better than naive", MASE > 1 means "worse".
   - The naive scaling denominator is always well-defined for non-constant series.

CONCLUSION: Use MASE as the primary metric for inflation forecasting.
Use MAE/RMSE as complementary metrics for interpretability within a single series.
Avoid MAPE for inflation data due to near-zero value problems.
""")